# Day 12. Query / Key / Value

**오늘의 목표**
- Query, Key, Value가 각각 어떤 역할인지 비유와 숫자로 설명할 수 있다
- 어제(Day 11) `softmax(X Xᵀ) X` 방식의 한계를 Q/K/V가 어떻게 해결하는지 안다
- `Q = X Wq`, `K = X Wk`, `V = X Wv`를 `nn.Linear`로 만들 수 있다
- Attention 한 번의 계산을 **shape를 찍어가며** 처음부터 끝까지 직접 따라갈 수 있다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()] if t.ndim == 1 else [r(row, d) for row in t]

tokens = ["나는", "오늘", "학교에", "간다"]

def show_table(weights, title):
    # (T, T) 가중치를 표로 출력
    print(title)
    print("        " + "".join(f"{t:>8s}" for t in tokens))
    for t, row in zip(tokens, weights):
        print(f"{t:>6s}  " + "".join(f"{v.item()*100:7.1f}%" for v in row))

## 1. 어제의 한계 다시 보기

Day 11에서는 Token 벡터 `X`만으로 Attention을 했습니다.

```text
weights = softmax(X Xᵀ)
output  = weights X
```

문제는 **벡터 하나가 모든 역할을 다 한다**는 점이었어요.
- "누구를 찾을지" 정할 때도 `X`
- "나를 찾게 할지" 정할 때도 `X`
- "실제로 넘겨줄 정보"도 `X`

그래서 자기 자신과의 Dot Product(`x · x`)가 가장 커지기 쉽고, 결국 **각 Token이 자기 자신만 많이 보게** 됩니다.

In [ ]:
X = torch.tensor([
    [1.0, 0.2, 0.0, 0.5],   # 나는
    [0.1, 1.0, 0.3, 0.0],   # 오늘
    [0.0, 0.4, 1.0, 0.2],   # 학교에
    [0.3, 0.0, 0.4, 1.0],   # 간다
])

show_table(F.softmax(X @ X.T, dim=-1), "softmax(X Xᵀ): 대각선(자기 자신)이 제일 큼")

하지만 우리가 원하는 건 이런 것입니다.

> "간다"는 **누가** 가는지 알고 싶다 → **"나는"** 을 봐야 한다

이걸 하려면 "찾는 쪽의 표현"과 "찾아지는 쪽의 표현"을 **따로** 둬야 합니다.
그게 Query와 Key예요.

## 2. 비유: 도서관 검색

| 역할 | 도서관 비유 | Attention에서 |
|---|---|---|
| **Query** | 내가 입력한 **검색어** | 이 Token이 **찾고 싶은 것** |
| **Key** | 책마다 붙은 **제목·태그** | 각 Token이 **"나는 이런 정보야"라고 내건 간판** |
| **Value** | 책의 **실제 내용** | 매칭되면 **실제로 가져갈 정보** |

검색 과정:
1. 검색어(Query)를 모든 책의 태그(Key)와 비교 → **Dot Product**
2. 비교 점수를 확률로 → **Softmax**
3. 점수가 높은 책의 **내용(Value)** 을 많이 가져옴 → **가중 평균**

포인트는 **태그(Key)와 내용(Value)이 다르다**는 것입니다. 태그로 찾고, 내용을 가져와요.

### 2-1. 숫자로 해보기

In [ ]:
# 태그 2칸: [동물, 요리]
keys = torch.tensor([
    [1.0, 0.0],   # 책 A: 동물 책
    [0.0, 1.0],   # 책 B: 요리 책
    [0.9, 0.1],   # 책 C: 동물 책 (요리가 조금)
])

# 내용 3칸: 태그와 전혀 다른 정보 (예: [고양이 정보, 파스타 정보, 강아지 정보])
values = torch.tensor([
    [10.0,  0.0,  0.0],   # 책 A 내용
    [ 0.0, 10.0,  0.0],   # 책 B 내용
    [ 0.0,  0.0, 10.0],   # 책 C 내용
])

query = torch.tensor([3.0, 0.0])          # 검색어: "동물" (강하게)

scores  = keys @ query                    # 검색어와 각 태그의 Dot Product
weights = F.softmax(scores, dim=-1)
result  = weights @ values                # 내용의 가중 평균

print("점수 (Query · Key)    :", r(scores))
print("가중치 (softmax)      :", r(weights))
print("가져온 내용 (가중 평균):", r(result, 2))

"동물"을 검색했더니 **동물 책 A와 C의 내용**(고양이·강아지 정보)이 대부분이고 요리 책 내용은 거의 없습니다.

- 매칭은 **2칸짜리 태그**로 했고
- 가져온 건 **3칸짜리 내용**입니다

**Key와 Value가 서로 다른 공간**에 있어도 된다는 게 핵심이에요.

## 3. Q/K로 "다른 Token 보기"가 가능해진다

이제 같은 문장에서, Token마다 **Query와 Key를 따로** 만들어 봅시다.
지금은 원리를 보려고 손으로 정한 값이에요. 실제로는 학습으로 만들어집니다.

- "간다"의 Query: **"주어를 찾고 있어"** → `[1, 0]`
- "나는"의 Key: **"나는 주어야"** → `[1, 0]`
- 나머지 Key는 주어가 아니라고 표시 → `[0, 1]` 근처

In [ ]:
Q_hand = torch.tensor([
    [0.0, 1.0],   # 나는   : 주변 맥락을 찾음
    [0.0, 1.0],   # 오늘   : 주변 맥락을 찾음
    [0.0, 1.0],   # 학교에 : 주변 맥락을 찾음
    [3.0, 0.0],   # 간다   : "주어 찾는 중!"
])
K_hand = torch.tensor([
    [1.0, 0.0],   # 나는   : "나는 주어야"
    [0.0, 1.0],   # 오늘
    [0.0, 1.0],   # 학교에
    [0.0, 1.0],   # 간다
])

show_table(F.softmax(Q_hand @ K_hand.T, dim=-1), "softmax(Q Kᵀ)")

마지막 줄을 보세요. **"간다"가 자기 자신이 아니라 "나는"을 가장 많이** 봅니다.

`X Xᵀ`에서는 불가능했던 일이에요. 찾는 표현(Query)과 찾아지는 표현(Key)을 분리했기 때문에 가능해졌습니다.

### 3-1. 비대칭: "내가 너를 보는 것" ≠ "네가 나를 보는 것"

`X Xᵀ`는 항상 **대칭**입니다. (`xᵢ · xⱼ = xⱼ · xᵢ`)
그래서 "간다 → 나는" 점수와 "나는 → 간다" 점수가 똑같을 수밖에 없어요.

`Q Kᵀ`는 `qᵢ · kⱼ`와 `qⱼ · kᵢ`가 다른 벡터끼리의 곱이라서 **대칭이 아닙니다.**

In [ ]:
S_xx = X @ X.T
S_qk = Q_hand @ K_hand.T

print("X Xᵀ 대칭인가?", torch.allclose(S_xx, S_xx.T))
print("Q Kᵀ 대칭인가?", torch.allclose(S_qk, S_qk.T))
print()
print(f"Q Kᵀ: 간다 → 나는 = {S_qk[3, 0].item():.1f},   나는 → 간다 = {S_qk[0, 3].item():.1f}")

언어에서는 관계가 대칭이 아닌 경우가 많습니다. "간다"는 주어 "나는"이 꼭 필요하지만, "나는"이 "간다"를 그만큼 필요로 하진 않죠.

## 4. Q, K, V는 어떻게 만드나? → `nn.Linear` 세 개

3절에서는 Q/K를 손으로 정했지만, 실제로는 **같은 입력 `X`에 서로 다른 Weight를 곱해서** 만듭니다.

```text
X
│
├─ Wq → Q    (찾고 싶은 것)
├─ Wk → K    (내가 내건 간판)
└─ Wv → V    (넘겨줄 정보)
```

Day 3의 6절에서 본 코드 그대로입니다.

In [ ]:
B, T, C = 2, 4, 16      # 문장 2개, Token 4개, 벡터 길이 16
head_size = 8           # Q/K/V 벡터 길이

query = nn.Linear(C, head_size, bias=False)
key   = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

x = torch.randn(B, T, C)
q, k, v = query(x), key(x), value(x)

print("x:", tuple(x.shape))
print("q:", tuple(q.shape))
print("k:", tuple(k.shape))
print("v:", tuple(v.shape))

- 세 Layer 모두 입력 `X`는 **같습니다**
- Weight가 다르니까 **결과가 다릅니다**
- 처음엔 랜덤이고, 학습(Day 7)을 하면서 "무엇을 찾을지 / 무엇을 내걸지 / 무엇을 넘길지"를 스스로 배웁니다

> 💡 `head_size`는 `C`와 달라도 됩니다. 2주차 마지막 날 Multi-Head에서 `head_size = C / head 개수`로 정하는 이유를 봅니다.

## 5. 처음부터 끝까지: Attention 한 번 (shape 따라가기)

커리큘럼 학습 원칙 3번 "직접 값을 출력한다"를 그대로 해봅니다.

In [ ]:
torch.manual_seed(42)
B, T, C, head_size = 2, 4, 16, 8

x = torch.randn(B, T, C)
query = nn.Linear(C, head_size, bias=False)
key   = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

# 1) Q, K, V 만들기
q = query(x)                                  # (B, T, head_size)
k = key(x)                                    # (B, T, head_size)
v = value(x)                                  # (B, T, head_size)
print("1) q, k, v      :", tuple(q.shape), tuple(k.shape), tuple(v.shape))

# 2) Attention Score: 모든 Token 쌍의 Q·K
k_t = k.transpose(-2, -1)                     # (B, head_size, T)
print("2) k 뒤집기      :", tuple(k_t.shape))
scores = q @ k_t                              # (B, T, T)
print("   scores = q@kᵀ :", tuple(scores.shape))

# 3) Softmax: 행마다 확률로
weights = F.softmax(scores, dim=-1)           # (B, T, T)
print("3) weights      :", tuple(weights.shape), "  각 행의 합:", r(weights[0].sum(dim=-1), 4))

# 4) Value의 가중 평균
out = weights @ v                             # (B, T, T) @ (B, T, head_size) → (B, T, head_size)
print("4) out = w @ v  :", tuple(out.shape))

**shape 요약**

```text
x        (B, T, C)
 ├ q     (B, T, head_size)
 ├ k     (B, T, head_size) → kᵀ (B, head_size, T)
 └ v     (B, T, head_size)
scores   (B, T, T)          ← q @ kᵀ : Token × Token 표
weights  (B, T, T)          ← softmax(dim=-1)
out      (B, T, head_size)  ← weights @ v
```

- `scores[b, i, j]` = b번째 문장에서 **i번째 Token(Query)이 j번째 Token(Key)과 얼마나 맞는가**
- `out[b, i]` = i번째 Token이 **모든 Token의 Value를 weights 비율로 섞어서 가져온 결과**

결국 Attention은 **Token 개수 `T`는 그대로 두고, 각 Token에 다른 Token들의 정보를 섞어 넣는** 연산입니다.

### 5-1. 가중치 히트맵

In [ ]:
w0 = weights[0].detach()

plt.figure(figsize=(5, 4))
plt.imshow(w0, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="가중치")
plt.xticks(range(T), tokens)
plt.yticks(range(T), tokens)
plt.xlabel("Key (참고하는 Token)")
plt.ylabel("Query (지금 Token)")
for i in range(T):
    for j in range(T):
        plt.text(j, i, f"{w0[i, j]*100:.0f}%", ha="center", va="center",
                 color="white" if w0[i, j] > 0.5 else "black", fontsize=10)
plt.title("첫 번째 문장의 Attention Weight (학습 전)", fontsize=12)
plt.tight_layout()
plt.show()

지금은 Weight가 랜덤이라 이 패턴에는 의미가 없습니다.
학습이 끝난 GPT에서는 이 표에 "간다 → 나는" 같은 **의미 있는 관계**가 나타납니다.

## 6. LLM과의 연결

오늘 만든 코드는 커리큘럼 6-4 **"Self-Attention 직접 구현"** 의 거의 전부입니다.
mini GPT에서는 이걸 `Head`라는 클래스로 묶습니다.

```python
class Head(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

    def forward(self, x):                       # x: (B, T, C)
        q, k, v = self.query(x), self.key(x), self.value(x)
        weights = F.softmax(q @ k.transpose(-2, -1), dim=-1)
        return weights @ v                      # (B, T, head_size)
```

아직 두 가지가 빠져 있어요.
- **내일 (Day 13)**: `scores`를 `√head_size`로 나누는 이유 (Scaled Dot-Product)
- **Day 14**: 미래 Token을 못 보게 하는 Causal Mask, 그리고 Head 여러 개 (Multi-Head)

## 7. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** 입력 벡터 길이 `C = 32`, `head_size = 8`일 때 Query/Key/Value Layer를 만드세요. (bias 없음)

In [ ]:
C, head_size = 32, 8

query = None  # TODO
key   = None  # TODO
value = None  # TODO

for name, layer in [("query", query), ("key", key), ("value", value)]:
    assert isinstance(layer, nn.Linear), f"{name}가 nn.Linear가 아니에요"
    assert tuple(layer.weight.shape) == (head_size, C), f"{name}의 크기를 다시 확인하세요"
    assert layer.bias is None, f"{name}에 bias가 있어요"
print("Q1 정답!")

**Q2.** batch가 있는 `q`, `k`로 Attention Score를 계산하세요. 결과는 `(B, T, T)`여야 합니다.

In [ ]:
torch.manual_seed(0)
q = torch.randn(3, 5, 8)    # (B, T, head_size)
k = torch.randn(3, 5, 8)

scores = None  # TODO

assert scores is not None and tuple(scores.shape) == (3, 5, 5), "shape가 (B, T, T)가 아니에요"
assert torch.allclose(scores[1], q[1] @ k[1].T), "값이 달라요. 어느 축을 뒤집었는지 확인하세요"
print("Q2 정답!")

**Q3.** Q2의 `scores`로 Attention 출력을 완성하세요.
먼저 결과 shape를 **예측**해서 적고, 그 다음 계산하세요.

In [ ]:
v = torch.randn(3, 5, 16)   # Value는 길이 16 (Key와 달라도 됨!)

predicted_shape = "?"   # TODO: 튜플로
weights = None          # TODO: scores를 확률로
out     = None          # TODO

assert predicted_shape == (3, 5, 16), "shape 예측을 다시 해보세요. v의 마지막 축을 보세요"
assert torch.allclose(weights.sum(dim=-1), torch.ones(3, 5)), "각 행의 합이 1이 아니에요. softmax 축을 확인하세요"
assert torch.allclose(out, weights @ v), "out 계산을 다시 확인하세요"
print("Q3 정답!", tuple(out.shape))

**Q4.** 아래 두 점수 표가 **대칭**(`S == S.T`)인지 실행하기 **전에** 예측하세요. `True` / `False`

In [ ]:
torch.manual_seed(1)
X  = torch.randn(4, 6)
Wq = torch.randn(6, 3)
Wk = torch.randn(6, 3)

S1 = X @ X.T                    # 어제 방식
S2 = (X @ Wq) @ (X @ Wk).T      # 오늘 방식

answer_S1 = "?"   # TODO: True 또는 False
answer_S2 = "?"   # TODO

assert answer_S1 == torch.allclose(S1, S1.T), "S1 예측이 틀렸어요. xᵢ·xⱼ와 xⱼ·xᵢ를 비교해보세요"
assert answer_S2 == torch.allclose(S2, S2.T), "S2 예측이 틀렸어요. qᵢ·kⱼ와 qⱼ·kᵢ를 비교해보세요"
print("Q4 정답!")

## 8. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Query, Key, Value는 각각 무엇인가? 도서관 비유로 설명해보세요.
2. 어제의 `softmax(X Xᵀ) X` 방식은 어떤 한계가 있었고, Q/K를 분리하면 왜 해결되는가?
3. `QKᵀ`는 왜 계산하는가? 결과 `(T, T)` 표의 `[i, j]` 칸은 무엇을 의미하는가?
4. Attention에서 V는 어떤 역할을 하는가? Key와 Value를 왜 따로 두는가?
5. `x: (B, T, C)`에서 출발해 Attention 출력까지 각 단계의 shape를 말해보세요.
6. `Wq`, `Wk`, `Wv`는 누가 정하는가?

**내 답:**

1.
2.
3.
4.
5.
6.